# Tests de validation MSUC

Ce notebook lance les deux scripts de validation du dossier `examples/`, 
qui verifient les proprietes theoriques de l'article (encadrement borne/optimum/politique, 
equivalence des formulations 3-bin et intervalle - Prop. 3), puis propose une section 
d'exploration interactive (section 4) pour tester une instance, un nombre de scenarios 
et une methode au choix.

A executer depuis la racine du depot (le kernel Julia active automatiquement ce projet).

## 1. Installation des dependances (une seule fois)

Installe les paquets listes dans `Project.toml` (JuMP, Gurobi.jl, CSV, DataFrames, NCDatasets, ...). 
Necessite une licence Gurobi valide sur la machine.

In [1]:
import Pkg
Pkg.activate(@__DIR__)  # active ce projet, quel que soit le kernel Julia choisi
Pkg.instantiate()

  Activating project at `~/Documents/MSUC`


In [3]:
using MSUC, Gurobi, Printf, Random

GRB_ENV = Gurobi.Env()
optimizer = () -> Gurobi.Optimizer(GRB_ENV)

Set parameter Username
Set parameter LicenseID to value 2818369
Academic license - for non-commercial use only - expires 2027-05-05


#2 (generic function with 1 method)

In [15]:
INSTANCE    = "Data/T-Ramp/10_0_1_w.nc4"  # chemin vers le fichier .nc4
T           = 24                           # horizon (nombre de pas de temps)
S           = 5                            # nombre de scenarios (support de l'innovation ARMA)
FORCE       = 1.0                          # amplitude de l'alea (0.0 = deterministe)
ITERATIONS  = 30                           # nombre d'iterations SDDP
PRINT_LEVEL = 1                            # 0 = silencieux, 1 = log complet de MSUC.train

# :threebin  -> formulation 3-bin  (Sect. 2 de l'article, cuts (13))
# :extended  -> formulation intervalle (Sect. 3 de l'article, cuts (39))
METHOD = :extended

:extended

In [ ]:
FORMULATIONS = Dict(
    :threebin => (MSUC.subproblem_builder_UC_def,     MSUC.SDDPFormulation,     "3-bin (13)"),
    :extended => (MSUC.subproblem_builder_UC_extended, MSUC.ExtendedFormulation, "intervalle (39)"),
)
builder, method, label = FORMULATIONS[METHOD]

instance = MSUC.parse_nc4(INSTANCE, optimizer, T; S = S)
@printf("instance %s : %d unites, %d intervalles, T = %d, S = %d, methode = %s\n",
        basename(INSTANCE), instance.N,
        sum(length(u.intervals) for u in instance.Thermalunits), T, S, label)

instance 10_0_1_w.nc4 : 10 unites, 2025 intervalles, T = 24, S = 5, methode = intervalle (39)



[450743] signal 15: Complété
in expression starting at none:1
__poll at /lib/x86_64-linux-gnu/libc.so.6 (unknown line)
_ZN3zmq15socket_poller_t4waitEP18zmq_poller_event_til at /home/mathis/.julia/artifacts/a95373603fb869dcd86b9f7191e57a4918ba6d2c/lib/libzmq.so (unknown line)
_ZN3zmq15proxy_steerableEPNS_13socket_base_tES1_S1_S1_ at /home/mathis/.julia/artifacts/a95373603fb869dcd86b9f7191e57a4918ba6d2c/lib/libzmq.so (unknown line)
zmq_proxy at /home/mathis/.julia/packages/ZMQ/yNY0H/src/bindings.jl:359 [inlined]
heartbeat_thread at /home/mathis/.julia/packages/IJulia/Vl5w1/src/heartbeat.jl:15
jlcapi_heartbeat_thread_6016 at /home/mathis/.julia/compiled/v1.12/IJulia/nfu7T_1aT8x.so (unknown line)
unknown function (ip: 0x7a4c26894a82) at /lib/x86_64-linux-gnu/libc.so.6
unknown function (ip: 0x7a4c269268df) at /lib/x86_64-linux-gnu/libc.so.6
unknown function (ip: (nil)) at (unknown file)
unknown function (ip: 0x7a4c268910d5) at /lib/x86_64-linux-gnu/libc.so.6
pthread_cond_wait at /lib/x86_6

: 

In [14]:
model = MSUC.PolicyGraph(
    builder, instance, FORCE, MSUC.Graph(T + 1), method;
    lower_bound = 0.0, optimizer = optimizer,
)

MSUC.train(model; iteration_limit = ITERATIONS, print_level = PRINT_LEVEL)

-------------------------------------------------------------------
         MSUC.jl (c) Oscar Dowson and contributors, 2017-25
-------------------------------------------------------------------
problem
  nodes           : 25
  state variables : 742
  scenarios       : 5.96046e+16
  existing cuts   : false
options
  solver          : serial mode
  risk measure    : MSUC.Expectation()
  sampling scheme : MSUC.InSampleMonteCarlo
subproblem structure
  VariableRef                             : [3510, 3514]
  AffExpr in MOI.EqualTo{Float64}         : [997, 1453]
  AffExpr in MOI.GreaterThan{Float64}     : [1, 11]
  AffExpr in MOI.LessThan{Float64}        : [40, 960]
  VariableRef in MOI.EqualTo{Float64}     : [1, 1]
  VariableRef in MOI.GreaterThan{Float64} : [2765, 2767]
  VariableRef in MOI.LessThan{Float64}    : [740, 740]
  VariableRef in MOI.ZeroOne              : [730, 730]
numerical stability report
  matrix range     [1e-01, 1e+03]
  objective range  [1e+00, 3e+03]
  bounds range 

### Resultats : trajectoire de la borne et cout de la politique simulee

In [9]:
log = model.most_recent_training_results.log
for l in log
    @printf("it %3d   borne = %.2f\n", l.iteration, l.bound)
end

it   1   borne = 58914.48
it   2   borne = 357938.91
it   3   borne = 373176.66
it   4   borne = 379926.29
it   5   borne = 397464.29
it   6   borne = 1613003.48
it   7   borne = 1631747.04
it   8   borne = 1637280.05
it   9   borne = 1650868.85
it  10   borne = 1758544.44
it  11   borne = 1760175.82
it  12   borne = 1760320.38
it  13   borne = 1775879.41
it  14   borne = 1776659.03
it  15   borne = 1778560.24
it  16   borne = 1780656.23
it  17   borne = 1780913.75
it  18   borne = 1782290.16
it  19   borne = 1791915.61
it  20   borne = 1799730.99


In [10]:
sim = MSUC.simulate_policy(model)
@printf("\ncout de la politique simulee = %.2f  (%d chemins, exhaustif = %s)\n",
        sim.cost, sim.npaths, sim.exhaustive)
@printf("borne finale = %.2f  ->  encadrement relatif = %.2e\n",
        log[end].bound, (sim.cost - log[end].bound) / abs(log[end].bound))


cout de la politique simulee = 2021536.39  (1000 chemins, exhaustif = false)
borne finale = 1799730.99  ->  encadrement relatif = 1.23e-01
